# Step 2: Combinations

# Mounting Google Drive

In [1]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


# Move to the Dataset Directory in My Drive

In [2]:
import os
os.chdir("/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025/Sample Data")
!pwd

/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025/Sample Data


# Combinations Code:

In [3]:
# Importing necessary packages
import matplotlib.pyplot as plt # for making plots / graphs
import pandas            as pd  # for reading the .csv file and related operations
import numpy             as np  # for working with arrays (multi-dimensional)
import numpy             as np
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm             import SVC
from sklearn.metrics         import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix, recall_score
from sklearn.preprocessing   import StandardScaler
from ast                     import literal_eval
from utils                   import *

# CARGAR LA BASE DE DATOS
file_path_train = f"ALL FEATURES OLDER - Pes Planus and Control JUNE 25 Train.xlsx"
file_path_test  = f"ALL FEATURES OLDER - Pes Planus and Control JUNE 25 Test.xlsx"
file_path_Step_1 = f"ML_Step_1.xlsx"

EXCEL_file_train = pd.ExcelFile(file_path_train)
EXCEL_file_test  = pd.ExcelFile(file_path_test)
Step_2_ML = {}

for sheet_name in EXCEL_file_train.sheet_names:

    # LEER EL DATA SET DE DATOS PRINCIPAL
    # Load the sheet into a DataFrame
    df = pd.read_excel(file_path_train, sheet_name=sheet_name)

    # df = df.drop(columns='Time_to_OHS')
    #ROM_features = list(df.filter(regex='_ROM'))
    #df = df.drop(columns=ROM_features)
    # now, the whole dataset csv dataset file is saved into `df` variable.
    # Perform Data Preprocessing
    # Label Encoding the class variables
    # Here, we replace the "Control" and "Flatfoot" keywords with 0 and 1 values, respectively.
    df["Group"] = df["Group"].replace({'Control': 0, 'Flatfoot': 1})
    # saving the target variables into `y` variable.
    Y_train = df.loc[:, "Group"].values
    # Perform Data Preprocessing- Data Standardization
    # Defining a Standard Scaler for scaling the values in the dataset
    # in the range of [-a, +a], i.e. scale values to a smaller range.

    # Define the different segments from dataset to be used.
    segments = {
        'AllF': df.loc[:, 'Pelv_Angle_Y_MAX_SW': 'Hip_Angle_Z_OHS'],
    }

    df_test = pd.read_excel(file_path_test, sheet_name=sheet_name)
    #ROM_features = list(df_test.filter(regex='_ROM'))
    # df_test = df_test.drop(columns='Time_to_OHS')
    #df_test = df_test.drop(columns=ROM_features)
    # now, the whole dataset csv dataset file is saved into `df` variable.
    # drop some subjects
    # df_test = df_test.drop([12,13], axis=0)
    # Remove the Columns: ["Index", "Side"]- These columns were not needed.
    # df_test = df_test.drop(["Participant", "Index", "Side"], axis=1)
    # saving the target variables into `y` variable.
    # Perform Data Preprocessing
    # Label Encoding the class variables
    # Here, we replace the "Control" and "Flatfoot" keywords with 0 and 1 values, respectively.
    df_test["Group"] = df_test["Group"].replace({'Control': 0, 'Flatfoot': 1})
    Y_test = df_test.loc[:, "Group"].values

    # Perform Data Preprocessing - Data Standardization
    # Defining a Standard Scaler for scaling the values in the dataset
    # in the range of [-a, +a], i.e. scale values to a smaller range.

    # Define the different segments from dataset to be used.
    segments_test = {
        'AllF': df_test.loc[:, 'Pelv_Angle_Y_MAX_SW': 'Hip_Angle_Z_OHS'],
    }

    # read features
    df_features = pd.read_excel(file_path_Step_1, sheet_name=sheet_name)

    # CV for Grid Search
    NS = 10
    RS = 0

    skf_cv = StratifiedKFold(n_splits = NS, shuffle=True, random_state = RS)

    # DF for test results
    df_results = pd.DataFrame(
        columns=['CV_Accuracy', 'Test_Accuracy', 'Sensitivity', '#Features', 'C', 'gamma', 'CV_slpit', 'Random_State', 'Feature_idx',
                'Confusion_Matrix', 'y_pred'])

    # Combinations Define
    C_set = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170]
    r_set = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]
    feature_len_list = np.arange(5, 20)
    print(len(C_set) * len(r_set) * len(feature_len_list))

    #df_analize = pd.DataFrame(
    #   columns=[])
    sc = StandardScaler()

    for s in range(len(df_features)):
        print("Ciclo S: ", s)
        for feature_len in feature_len_list:
            # Get the Feature List index...
            res = np.array(literal_eval(df_features.iloc[:, -1][s]))

            sfs_feature_idx = res[:feature_len]
            # print(sfs_feature_idx)
            # Filter Training Data...
            X_train = segments["AllF"].iloc[:, sfs_feature_idx].values
            X_train = sc.fit_transform(X_train)
            # Filter Testing Data
            X_test = segments_test["AllF"].iloc[:, sfs_feature_idx].values
            X_test = sc.transform(X_test)
            for C_val in C_set:
                for r_val in r_set:

                    # Defining the SVM...
                    svm = SVC(kernel='rbf', C=C_val, gamma=r_val, verbose=False)
                    y_true_unflat_list, y_pred_unflat_list = [], []

                    for train_idx, test_idx in skf_cv.split(X_train, Y_train):
                        x_train, y_train = X_train[train_idx], Y_train[train_idx]
                        x_test, y_test = X_train[test_idx], Y_train[test_idx]

                        svm.fit(x_train, y_train)

                        y_pred = svm.predict(x_test)

                        y_true_unflat_list.append(y_test[:])
                        y_pred_unflat_list.append(y_pred[:])

                    y_true_list = [item for sublist in y_true_unflat_list for item in sublist]
                    y_pred_list = [item for sublist in y_pred_unflat_list for item in sublist]

                    svm.fit(X_train, Y_train)
                    y_pred = svm.predict(X_test)
                    if get_sensitivity(Y_test, y_pred) >= 0.5 and get_specificity(Y_test, y_pred) > 0.5:
                        if accuracy_score(y_true_list, y_pred_list) >= 0.5:
                            # print(get_sensitivity(Y_test, y_pred))
                            df_results = pd.concat([
                                df_results,
                                pd.DataFrame([[
                                    accuracy_score(y_true_list, y_pred_list),
                                    accuracy_score(Y_test, y_pred),
                                    get_sensitivity(Y_test, y_pred),
                                    len(sfs_feature_idx),
                                    C_val,
                                    r_val,
                                    NS,
                                    RS,
                                    s,
                                    confusion_matrix(Y_test, y_pred),
                                    y_pred]],
                                columns=['CV_Accuracy', 'Test_Accuracy', 'Sensitivity', '#Features', 'C', 'gamma',
                                        'CV_slpit',     'Random_State', 'Feature_idx', 'Confusion_Matrix', 'y_pred'])], ignore_index=True)

                            # print("{:.3f}, {:.3f}, {:.3f}, {}, {}, {}, {}".format(accuracy_score(y_true_list, y_pred_list), accuracy_score(Y_test, y_pred), get_sensitivity(Y_test, y_pred), len(res), C_val, r_val, s))
                            # print(confusion_matrix(Y_test, y_pred))
                            # print(y_pred)
                            # print("\n###################################################\n")
            print("Feature_idx: ", s, "#Features: ", len(sfs_feature_idx))

    Step_2_ML[sheet_name] = df_results


file_name = 'Step2_Results_v2.xlsx'
# Save all sheets to a new Excel file
with pd.ExcelWriter(file_name) as writer:
    for sheet_name, df_result in Step_2_ML.items():
        df_result.to_excel(writer, sheet_name=sheet_name, index=True)

/tmp/ipython-input-3-3575116396.py:37: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Group"] = df["Group"].replace({'Control': 0, 'Flatfoot': 1})
/tmp/ipython-input-3-3575116396.py:62: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test["Group"] = df_test["Group"].replace({'Control': 0, 'Flatfoot': 1})


2430
Ciclo S:  0
Feature_idx:  0 #Features:  5


/tmp/ipython-input-3-3575116396.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([


Feature_idx:  0 #Features:  6
Feature_idx:  0 #Features:  7
Feature_idx:  0 #Features:  8
Feature_idx:  0 #Features:  9
Feature_idx:  0 #Features:  10
Feature_idx:  0 #Features:  11
Feature_idx:  0 #Features:  12
Feature_idx:  0 #Features:  13
Feature_idx:  0 #Features:  14
Feature_idx:  0 #Features:  15
Feature_idx:  0 #Features:  16
Feature_idx:  0 #Features:  17
Feature_idx:  0 #Features:  18
Feature_idx:  0 #Features:  19
Ciclo S:  1
Feature_idx:  1 #Features:  5
Feature_idx:  1 #Features:  6
Feature_idx:  1 #Features:  7
Feature_idx:  1 #Features:  8
Feature_idx:  1 #Features:  9
Feature_idx:  1 #Features:  10
Feature_idx:  1 #Features:  11
Feature_idx:  1 #Features:  12
Feature_idx:  1 #Features:  13
Feature_idx:  1 #Features:  14
Feature_idx:  1 #Features:  15
Feature_idx:  1 #Features:  16
Feature_idx:  1 #Features:  17
Feature_idx:  1 #Features:  18
Feature_idx:  1 #Features:  19
Ciclo S:  2
Feature_idx:  2 #Features:  5
Feature_idx:  2 #Features:  6
Feature_idx:  2 #Features: